# 03 — Data Processing

**Goal:** turn the engineered data (from `02_feature_engineering`) into model-ready inputs.

| Step | Linear (OLS) | Regularized linear (Ridge / Lasso) | Trees (RF / GBM) |
|---|---|---|---|
| Scale numeric stats | `StandardScaler` | `StandardScaler` | none |
| Era / decade dummies | drop one baseline per group + eras that duplicate decades | keep all | keep all |
| `teamID` | one-hot, drop one (optional) | one-hot (optional) | one-hot (optional) |

The preprocessors are built here **unfitted** and saved. Each one is fitted inside a `Pipeline` during modeling, so scaling statistics are learned from the training folds only (no leakage into validation).

Steps:
1. Load engineered data & define column groups
2. Decide whether to use `teamID`
3. Choose baseline dummies for OLS
4. Build the preprocessors
5. Verify the outputs (scaling, collinearity)
6. Sanity-check with cross-validation
7. Save

In [1]:
import json
import joblib
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, GroupKFold, cross_val_score

# works whether the kernel starts in notebooks/, Model-3-Project/ or DSAI/
DATA_DIR = next(p for p in [Path('../Data'), Path('Data'), Path('Model-3-Project/Data')]
                if (p / 'train_features.csv').exists())
ARTIFACT_DIR = DATA_DIR.parent / 'artifacts'
ARTIFACT_DIR.mkdir(exist_ok=True)

train = pd.read_csv(DATA_DIR / 'train_features.csv')
pred = pd.read_csv(DATA_DIR / 'predict_features.csv')
fe_config = json.loads((ARTIFACT_DIR / 'engineered_features.json').read_text())
print('train:', train.shape, '| predict:', pred.shape)

train: (1412, 75) | predict: (400, 73)


## 1. Column groups

In [2]:
TARGET = 'W'
ID_COL = 'ID'
NON_FEATURES = [TARGET, ID_COL, 'yearID', 'decade_label']   # target, identifier, analysis-only, label-only

ERA_COLS = [c for c in train if c.startswith('era_')]
DECADE_COLS = [c for c in train if c.startswith('decade_') and c != 'decade_label']
FLAG_COLS = ERA_COLS + DECADE_COLS
CAT_COLS = ['teamID']
NUM_COLS = [c for c in train if c not in NON_FEATURES + FLAG_COLS + CAT_COLS]

ENGINEERED = fe_config['engineered']
assert set(ENGINEERED) <= set(NUM_COLS)
print(f'numeric ({len(NUM_COLS)}, of which {len(ENGINEERED)} engineered):', NUM_COLS)
print(f'era flags ({len(ERA_COLS)}), decade flags ({len(DECADE_COLS)}), categorical: {CAT_COLS}')

numeric (50, of which 25 engineered): ['G', 'R', 'AB', 'H', '2B', '3B', 'HR', 'BB', 'SO', 'SB', 'RA', 'ER', 'ERA', 'CG', 'SHO', 'SV', 'IPouts', 'HA', 'HRA', 'BBA', 'SOA', 'E', 'DP', 'FP', 'mlb_rpg', 'RD', 'RD_pg', 'R_pg', 'RA_pg', 'log_R_RA', 'pyth_W', 'pythpat_W', 'SV_pg', 'SHO_pg', 'CG_pg', 'IP_pg', 'R_rel', 'RA_rel', 'OBP', 'SLG', 'OPS', 'ISO', 'RC_pg', 'WHIP', 'K9', 'BB9', 'HR9', 'FIP_core', 'UER', 'E_pg']
era flags (8), decade flags (12), categorical: ['teamID']


In [3]:
# every column must be numeric (or the one categorical) and present in both files, with no missing values
all_features = NUM_COLS + FLAG_COLS + CAT_COLS
assert set(all_features) <= set(pred.columns)
assert train[all_features].isna().sum().sum() == 0 and pred[all_features].isna().sum().sum() == 0
assert train[NUM_COLS + FLAG_COLS].dtypes.map(pd.api.types.is_numeric_dtype).all()
assert set(train[FLAG_COLS].stack().unique()) <= {0, 1}

## 2. Should `teamID` be a feature?
The data dictionary allows `teamID` as a categorical feature. But team strength is already captured by the stats, codes change when franchises move (`KC1`, `ML1`, `ANA`/`LAA`), and some teams have only a handful of rows. We compare 5-fold CV error with and without it.

In [4]:
print(train['teamID'].value_counts().describe()[['min', '50%', 'max']])

cv = KFold(n_splits=5, shuffle=True, random_state=42)

def cv_rmse(model, X, y):
    scores = -cross_val_score(model, X, y, cv=cv, scoring='neg_root_mean_squared_error')
    return scores.mean(), scores.std()

def quick_prep(use_team, scale):
    steps = [('num', StandardScaler() if scale else 'passthrough', NUM_COLS),
             ('flags', 'passthrough', FLAG_COLS)]
    if use_team:
        steps.append(('team', OneHotEncoder(handle_unknown='ignore'), CAT_COLS))
    return ColumnTransformer(steps)

rows = []
for use_team in (False, True):
    for name, model, scale in [('Ridge', RidgeCV(alphas=np.logspace(-3, 3, 50)), True),
                               ('RandomForest', RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1), False)]:
        pipe = Pipeline([('prep', quick_prep(use_team, scale)), ('model', model)])
        mean, std = cv_rmse(pipe, train, train[TARGET])
        rows.append({'model': name, 'teamID': use_team, 'CV RMSE': round(mean, 3), '± std': round(std, 3)})
pd.DataFrame(rows).pivot(index='model', columns='teamID', values='CV RMSE')

min     3.0
50%    28.0
max    73.0
Name: count, dtype: float64


teamID,False,True
model,,
RandomForest,3.905,3.909
Ridge,3.410,3.437


Adding `teamID` doesn't lower the error for either model, so it's **left out by default**. Flip `USE_TEAM_ID` below to try it again later.

In [5]:
USE_TEAM_ID = False
CAT_FEATURES = CAT_COLS if USE_TEAM_ID else []
FEATURES = NUM_COLS + FLAG_COLS + CAT_FEATURES
print(f'{len(FEATURES)} features')

70 features


## 3. Baseline dummies for OLS
Each row has exactly one era flag and one decade flag, so each group sums to 1 and duplicates the intercept (the *dummy variable trap*). Plain linear regression needs one flag per group dropped; that category becomes the baseline the others are compared against.

The choice doesn't change predictions, only how coefficients read. We use the **most common category** in each group, so the comparisons are against a well-estimated group.

In [6]:
era_counts = train[ERA_COLS].sum().sort_values(ascending=False)
decade_counts = train[DECADE_COLS].sum().sort_values(ascending=False)
BASELINE_COLS = [era_counts.index[0], decade_counts.index[0]]

print('era rows:\n', era_counts.to_string(), '\n')
print('decade rows:\n', decade_counts.to_string(), '\n')
print('OLS baselines (dropped):', BASELINE_COLS)

era rows:
 era_7    284
era_6    244
era_2    218
era_5    205
era_4    169
era_1    130
era_8    125
era_3     37 

decade rows:
 decade_2000    197
decade_1970    162
decade_1980    149
decade_1990    134
decade_2010    125
decade_1960    121
decade_1950    108
decade_1920    107
decade_1930     92
decade_1940     87
decade_1910     81
decade_1900     49 

OLS baselines (dropped): ['era_7', 'decade_2000']


### Eras that duplicate decades
Some eras line up exactly with decades in this data. These are also exact linear combinations, so OLS can't use them either. We detect them: an era is redundant if it equals the sum of the decade flags it overlaps.

In [7]:
REDUNDANT_ERAS = []
for era in ERA_COLS:
    overlapping = [d for d in DECADE_COLS if (train.loc[train[era] == 1, d] == 1).any()]
    if all((df[era] == df[overlapping].sum(axis=1)).all() for df in (train, pred)):
        REDUNDANT_ERAS.append(era)
        print(f'{era} == ' + ' + '.join(overlapping))

# engineered features that are exact linear combinations of others (RD = R − RA, etc.)
EXACT_COMBOS = fe_config['exact_combos']
OLS_DROP_COLS = BASELINE_COLS + REDUNDANT_ERAS + EXACT_COMBOS
print('OLS drops:', OLS_DROP_COLS)

era_1 == decade_1900 + decade_1910
era_8 == decade_2010
OLS drops: ['era_7', 'decade_2000', 'era_1', 'era_8', 'RD', 'RD_pg', 'UER', 'OPS', 'FIP_core']


`era_8` (2010–2016) is exactly the 2010s, and `era_1` (1904–1917) is exactly 1900s + 1910s because the data has no 1918–1919 seasons. Dropping them loses nothing: the decade flags already carry that information.

The engineered features `RD`, `RD_pg`, `UER`, `OPS` and `FIP_core` are exact linear combinations of other columns (from `02_feature_engineering`), so OLS drops them too.

## 4. Build the preprocessors

In [8]:
def make_preprocessor(kind):
    """kind: 'ols' (scaled, baselines + redundant eras + exact combos dropped), 'regularized' (scaled, all flags), 'tree' (unscaled, all flags)."""
    if kind not in ('ols', 'regularized', 'tree'):
        raise ValueError(kind)
    drop = OLS_DROP_COLS if kind == 'ols' else []
    flags = [c for c in FLAG_COLS if c not in drop]
    nums = [c for c in NUM_COLS if c not in drop]
    steps = [('num', 'passthrough' if kind == 'tree' else StandardScaler(), nums),
             ('flags', 'passthrough', flags)]
    if CAT_FEATURES:
        steps.append(('cat', OneHotEncoder(handle_unknown='ignore',
                                           drop='first' if kind == 'ols' else None,
                                           sparse_output=False), CAT_FEATURES))
    return ColumnTransformer(steps, verbose_feature_names_out=False)

preprocessors = {kind: make_preprocessor(kind) for kind in ('ols', 'regularized', 'tree')}
preprocessors['regularized']

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('flags', ...)]"
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. ``""{feature_name}__{transformer_name}""``. See :meth:`str.format` method from the standard library for more info... versionadded:: 1.0.. versionchanged:: 1.6 `verbose_feature_names_out` can be a callable or a string to be formatted.",False
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. T

## 5. Verify the outputs
Fitted on the full training set **only to inspect them**. During modeling they're fitted inside each CV fold.

In [9]:
X_train = train[FEATURES]
for kind, prep in preprocessors.items():
    Xt = prep.fit_transform(X_train)
    print(f'{kind:12} -> {Xt.shape[1]} columns')

ols          -> 61 columns
regularized  -> 70 columns
tree         -> 70 columns


In [10]:
# scaled numeric columns should have mean ~0 and std ~1; flags stay 0/1
prep = preprocessors['regularized']
Xt = pd.DataFrame(prep.fit_transform(X_train), columns=prep.get_feature_names_out())
Xt[NUM_COLS].agg(['mean', 'std']).T.round(3).head(10)

,mean,std
G,-0.0,1.0
R,0.0,1.0
AB,0.0,1.0
H,0.0,1.0
2B,-0.0,1.0
3B,-0.0,1.0
HR,0.0,1.0
BB,-0.0,1.0
SO,0.0,1.0
SB,-0.0,1.0


In [11]:
# predict.csv goes through the same fitted transformer: same columns, same order,
# scaled with the TRAINING mean/std (so its means won't be exactly 0)
Xp = pd.DataFrame(prep.transform(pred[FEATURES]), columns=prep.get_feature_names_out())
assert list(Xp.columns) == list(Xt.columns)
Xp[NUM_COLS].agg(['mean', 'std']).T.round(3).head(5)

,mean,std
G,-0.025,1.022
R,0.022,1.015
AB,-0.011,1.084
H,0.014,1.040
2B,0.007,1.006


### Collinearity check
A design matrix (features + intercept column) has full rank when no column is an exact combination of others. Keeping everything leaves it rank-deficient by 9: one per flag group summing to 1, the two eras that duplicate decades, and the five exact-combination features. The OLS preprocessor removes all nine.

In [12]:
def rank_report(kind):
    Xt = preprocessors[kind].fit_transform(X_train)
    design = np.column_stack([np.ones(len(Xt)), Xt])
    return {'columns (incl. intercept)': design.shape[1], 'rank': np.linalg.matrix_rank(design),
            'redundant columns': design.shape[1] - np.linalg.matrix_rank(design)}

ranks = pd.DataFrame({kind: rank_report(kind) for kind in ('regularized', 'ols')}).T
assert ranks.loc['ols', 'redundant columns'] == 0
ranks

,columns (incl. intercept),rank,redundant columns
regularized,71,62,9
ols,62,62,0


The OLS matrix has no exact redundancy. The regularized/tree matrices keep all flags on purpose: Ridge's penalty gives a unique solution anyway, and trees don't solve equations.

## 6. Sanity check with cross-validation
Each preprocessor wrapped in a pipeline with a simple model, to confirm everything runs end to end and give baseline scores. Scored two ways: random 5-fold, and **leave one decade out** (matches the competition's cross-era goal). Proper model tuning comes in the modeling notebook.

In [13]:
baseline_models = {
    'OLS': ('ols', LinearRegression()),
    'Ridge': ('regularized', RidgeCV(alphas=np.logspace(-3, 3, 50))),
    'RandomForest': ('tree', RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)),
}
decade_cv = GroupKFold(n_splits=train['decade_label'].nunique())
results = []
for name, (kind, model) in baseline_models.items():
    pipe = Pipeline([('prep', make_preprocessor(kind)), ('model', model)])
    mean, std = cv_rmse(pipe, X_train, train[TARGET])
    decade = -cross_val_score(pipe, X_train, train[TARGET], cv=decade_cv, groups=train['decade_label'],
                              scoring='neg_root_mean_squared_error').mean()
    results.append({'model': name, 'preprocessor': kind, 'random 5-fold RMSE': round(mean, 3), '± std': round(std, 3),
                    'leave-decade-out RMSE': round(decade, 3)})
pd.DataFrame(results)

,model,preprocessor,random 5-fold RMSE,± std,leave-decade-out RMSE
0,OLS,ols,3.462,0.064,3.838
1,Ridge,regularized,3.410,0.061,3.696
2,RandomForest,tree,3.905,0.118,4.063


## 7. Save

In [14]:
config = {
    'target': TARGET, 'id_col': ID_COL, 'features': FEATURES,
    'num_cols': NUM_COLS, 'flag_cols': FLAG_COLS, 'cat_features': CAT_FEATURES,
    'ols_drop_cols': OLS_DROP_COLS, 'use_team_id': USE_TEAM_ID,
}
(ARTIFACT_DIR / 'feature_config.json').write_text(json.dumps(config, indent=2))
joblib.dump({kind: make_preprocessor(kind) for kind in preprocessors}, ARTIFACT_DIR / 'preprocessors.joblib')
print('Saved:', ARTIFACT_DIR / 'feature_config.json', '|', ARTIFACT_DIR / 'preprocessors.joblib')

Saved: ../artifacts/feature_config.json | ../artifacts/preprocessors.joblib


### Using these in the modeling notebook
```python
import json, joblib
config = json.loads(Path('../artifacts/feature_config.json').read_text())
preprocessors = joblib.load('../artifacts/preprocessors.joblib')   # unfitted

pipe = Pipeline([('prep', preprocessors['regularized']), ('model', RidgeCV(...))])
pipe.fit(train[config['features']], train[config['target']])
pipe.predict(pred[config['features']])
```

## Summary
| Step | Result |
|---|---|
| Column groups | raw + engineered numeric stats, 8 era + 12 decade flags, `teamID` |
| `teamID` | excluded by default (no CV gain); toggle `USE_TEAM_ID` |
| OLS drops | most common era & decade (baselines), `era_1`, `era_8` (duplicate decades), `RD`, `RD_pg`, `UER`, `OPS`, `FIP_core` (exact combos) |
| Preprocessors | `ols`, `regularized`, `tree`, saved unfitted to `artifacts/` |

**Next:** model tuning and comparison.